In [15]:
import sys
sys.path.append('/host/d/Github')
import nibabel as nb
import glob
import os
import glob
import lpips
import torch
import numpy as np
import pandas as pd
from skimage.metrics import structural_similarity

import Diffusion_denoising_thin_slice.functions_collection as ff
import Diffusion_denoising_thin_slice.Build_lists.Build_list as Build_list
import Diffusion_denoising_thin_slice.Data_processing as Data_processing

In [16]:
build_sheet =  Build_list.Build(os.path.join('/host/e/D/Data/low_dose_CT/Patient_lists/mayo_low_dose_CT_gaussian_simulation_highnoise_v2.xlsx'))
batch_list, patient_id_list, random_num_list,noise_file_all_list, noise_file_odd_list, noise_file_even_list, ground_truth_file_list, slice_num_list = build_sheet.__build__(batch_list = ['test']) 
n = ff.get_X_numbers_in_interval(total_number = patient_id_list.shape[0],start_number = 0,end_number = 1, interval = 1)

In [17]:
def calc_mae_with_ref_window(img, ref, vmin, vmax):
    maes = []
    for slice_num in range(0, img.shape[-1]):
        slice_img = img[:,:,slice_num]
        slice_ref = ref[:,:,slice_num]
        mask = np.where((slice_ref >= vmin) & (slice_ref <= vmax), 1, 0)
        mae = np.sum(np.abs(slice_img - slice_ref) * mask) / np.sum(mask)
        maes.append(mae)

    return np.mean(maes), np.std(maes)

In [18]:
def calc_ssim_with_ref_window(img, ref, vmin, vmax):

    ssims = []
    for slice_num in range(0, img.shape[-1]):
        slice_img = img[:,:,slice_num]
        slice_ref = ref[:,:,slice_num]
        mask = np.where((slice_ref >= vmin) & (slice_ref <= vmax), 1, 0)
        _, ssim_map = structural_similarity(slice_img, slice_ref, data_range=vmax - vmin, full=True)
        ssim = np.sum(ssim_map * mask) / np.sum(mask)
        ssims.append(ssim)

    return np.mean(ssims), np.std(ssims)

In [20]:
def calc_lpips(imgs1, imgs2, vmin, vmax):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    loss_fn = lpips.LPIPS().to(device)
    
    lpipss = []
    for slice_num in range(0, imgs1.shape[-1]):
        slice1 = imgs1[:,:,slice_num]
        slice2 = imgs2[:,:,slice_num]

        slice1 = np.clip(slice1, vmin, vmax).astype(np.float32)
        slice2 = np.clip(slice2, vmin, vmax).astype(np.float32)

        slice1 = (slice1 - vmin) / (vmax - vmin) * 2 - 1
        slice2 = (slice2 - vmin) / (vmax - vmin) * 2 - 1

        slice1 = np.stack([slice1, slice1, slice1], axis=-1)
        slice2 = np.stack([slice2, slice2, slice2], axis=-1)
        # print('after stack, slice1 shape:', slice1.shape, ' slice2 shape:', slice2.shape)

        slice1 = np.transpose(slice1, (2, 0, 1))[np.newaxis, ...]
        slice2 = np.transpose(slice2, (2, 0, 1))[np.newaxis, ...]
        # print('after transpose, slice1 shape:', slice1.shape, ' slice2 shape:', slice2.shape)

        slice1 = torch.from_numpy(slice1).to(device)
        slice2 = torch.from_numpy(slice2).to(device)

        lpips_val = loss_fn(slice1, slice2)
        lpipss.append(lpips_val.item())

      

    return np.mean(lpipss), np.std(lpipss)


### main

In [24]:
## metric calculations
from genericpath import isfile
import shutil

results = []
save_collection = False

NFE_list = [30,50]

for i in range(0,n.shape[0]):
    patient_id = patient_id_list[n[i]]
    random_n = random_num_list[n[i]]
    print(patient_id,  random_n)


    # reference image
    gt_file = os.path.join('/host/e/D/Data/low_dose_CT/nii_imgs', patient_id,  'img.nii.gz')
    gt_img = nb.load(gt_file).get_fdata()[...,150:200]
    gt_img = np.clip(gt_img, -200,250)
    gt_mean = np.mean(gt_img[(gt_img>-160) & (gt_img<240)])

    # noisy image
    condition_file = os.path.join('/host/e/D/Data/low_dose_CT/simulation_highnoise_v2', patient_id,'gaussian_random_'+str(random_n), 'recon_all.nii.gz')
    condition_img = nb.load(condition_file).get_fdata()[...,150:200]
    condition_img = np.clip(condition_img, -200,250)
    condition_mean = np.mean(condition_img[(gt_img>-160) & (gt_img<240)])

#     # noise2noise
#     noise2noise_file = os.path.join('/host/d/projects/denoising/models/noise2noise_mayo_highnoise/pred_images_input_both', patient_id,'random_'+str(random_n), 'epoch50/pred_img.nii.gz')
#     noise2noise_img = nb.load(noise2noise_file).get_fdata()

#     # noise2score
#     noise2score_file = os.path.join('/host/d/projects/denoising/models/noise2score_mayo/pred_images_highnoise', patient_id,'pred_img_mayo.nii.gz')
#     noise2score_img = nb.load(noise2score_file).get_fdata()

#    # DDM2_firststep
#     ddm2_first_file = os.path.join('/host/d/projects/denoising/models/DDM2_mayo/pred_images_highnoise', patient_id,'ddm2_first_step.nii.gz')
#     ddm2_first_img = nb.load(ddm2_first_file).get_fdata()

#     # DDM2_finalstep
#     ddm2_final_file = os.path.join('/host/d/projects/denoising/models/DDM2_mayo/pred_images_highnoise', patient_id, 'ddm2_final.nii.gz')
#     ddm2_final_img = nb.load(ddm2_final_file).get_fdata()


#     #supervised method
#     supervised_file = os.path.join('/host/d/projects/denoising/models/supervised_poisson_mayo_highnoise/pred_images_input_all',patient_id,  'random_'+str(random_n), 'epoch100_1/pred_img.nii.gz')
#     supervised_img = nb.load(supervised_file).get_fdata()
    # if save_collection:
    #     shutil.copy(supervised_file, os.path.join(save_folder, 'supervised_img.nii.gz'))

    # our method: 1 inference
    # our_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_mayo_highnoise/', 'pred_images_input_both', patient_id,'random_'+str(random_n), 'epoch165avg/pred_img_scans1.nii.gz')
    # our_img = nb.load(our_file).get_fdata()
    # our_mean = np.mean(our_img[(gt_img>-160) & (gt_img<240)])

    # # our method: 20 inference
    # our_avg20_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_mayo_highnoise/', 'pred_images_input_both', patient_id,'random_'+str(random_n), 'epoch165avg/pred_img_scans20.nii.gz')
    # our_avg20_img = nb.load(our_avg20_file).get_fdata()
    # our_avg20_mean = np.mean(our_avg20_img[(gt_img>-160) & (gt_img<240)])

    ## calculate metrics
    # MAE
    vmin = -160; vmax = 240
    mae_condition, mae_condition_std = calc_mae_with_ref_window(condition_img, gt_img, vmin, vmax)
    # mae_noise2noise, mae_noise2noise_std = calc_mae_with_ref_window(noise2noise_img, gt_img, vmin, vmax)
    # mae_noise2score, mae_noise2score_std = calc_mae_with_ref_window(noise2score_img, gt_img, vmin, vmax)
    # mae_ddm2_final, mae_ddm2_final_std = calc_mae_with_ref_window(ddm2_final_img, gt_img, vmin, vmax)
    # mae_supervised, mae_supervised_std = calc_mae_with_ref_window(supervised_img, gt_img, vmin, vmax)
    # mae_our, mae_our_std = calc_mae_with_ref_window(our_img, gt_img, vmin, vmax)
    # mae_our_avg20, mae_our_avg20_std = calc_mae_with_ref_window(our_avg20_img, gt_img, vmin, vmax)

    # # SSIM
    ssim_condition, ssim_condition_std = calc_ssim_with_ref_window(condition_img, gt_img, vmin, vmax)
    # ssim_noise2noise, ssim_noise2noise_std = calc_ssim_with_ref_window(noise2noise_img, gt_img, vmin, vmax)
    # ssim_noise2score, ssim_noise2score_std = calc_ssim_with_ref_window(noise2score_img, gt_img, vmin, vmax)
    # ssim_ddm2_final, ssim_ddm2_final_std = calc_ssim_with_ref_window(ddm2_final_img, gt_img, vmin, vmax)
    # ssim_supervised, ssim_supervised_std = calc_ssim_with_ref_window(supervised_img, gt_img, vmin, vmax)
    # ssim_our, ssim_our_std = calc_ssim_with_ref_window(our_img, gt_img, vmin, vmax)
    # ssim_our_avg20, ssim_our_avg20_std = calc_ssim_with_ref_window(our_avg20_img, gt_img, vmin, vmax)

    # # LPIPS
    lpips_condition, _ = calc_lpips(condition_img, gt_img, vmin, vmax)
    # lpips_noise2noise, _ = calc_lpips(noise2noise_img, gt_img, vmin, vmax)
    # lpips_noise2score, _ = calc_lpips(noise2score_img, gt_img, vmin, vmax)
    # lpips_ddm2_final, _ = calc_lpips(ddm2_final_img, gt_img, vmin, vmax)
    # lpips_supervised, _ = calc_lpips(supervised_img, gt_img, vmin, vmax)
    # lpips_our, _ = calc_lpips(our_img, gt_img, vmin, vmax)
    # lpips_our_avg20, _ = calc_lpips(our_avg20_img, gt_img, vmin, vmax)

    print('mae_condition, ssim_condition, lpips_condition: ', mae_condition, ssim_condition, lpips_condition)
    # print('mae_ours, ssim_ours, lpips_ours: ', mae_our, ssim_our, lpips_our)
    # print('mae_our_avg20, ssim_our_avg20, lpips_our_avg20: ', mae_our_avg20, ssim_our_avg20, lpips_our_avg20)


    per_patient_results = [patient_id, random_n, mae_condition, ssim_condition, lpips_condition]#, mae_noise2noise, ssim_noise2noise, lpips_noise2noise, mae_noise2score, ssim_noise2score, lpips_noise2score, mae_ddm2_final, ssim_ddm2_final, lpips_ddm2_final, mae_supervised, ssim_supervised, lpips_supervised]

    for NFE in NFE_list:
        print('NFE is: ', NFE)
        # our method (unsupervised), 1 inference
        unsupervised_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_mayo_highnoise_predict_noise_bias/', 'pred_images_NFE'+str(NFE)+'_ETA_1.0', patient_id,'random_'+str(random_n), 'epoch285_1/pred_img.nii.gz')
        unsupervised_img = nb.load(unsupervised_file).get_fdata()
        unsupervised_mean = np.mean(unsupervised_img[(gt_img>-160) & (gt_img<240)])

       
        # our method (unsupervised), 10 inference
        unsupervised_avg10_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_mayo_highnoise_predict_noise_bias/', 'pred_images_NFE'+str(NFE)+'_ETA_1.0', patient_id,'random_'+str(random_n), 'epoch285avg/pred_img_scans10.nii.gz')
        unsupervised_avg10_img = nb.load(unsupervised_avg10_file).get_fdata()
        unsupervised_avg10_mean = np.mean(unsupervised_avg10_img[(gt_img>-160) & (gt_img<240)])
        

        # our method (unsupervised), beta = 0, 20 inference
        unsupervised_avg20_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_mayo_highnoise_predict_noise_bias/', 'pred_images_NFE'+str(NFE)+'_ETA_1.0', patient_id,'random_'+str(random_n), 'epoch285avg/pred_img_scans20.nii.gz')
        unsupervised_avg20_img = nb.load(unsupervised_avg20_file).get_fdata()
        unsupervised_avg20_mean = np.mean(unsupervised_avg20_img[(gt_img>-160) & (gt_img<240)])


        mae_unsupervised, mae_unsupervised_std = calc_mae_with_ref_window(unsupervised_img, gt_img, vmin, vmax)
        mae_unsupervised_avg10, mae_unsupervised_avg10_std = calc_mae_with_ref_window(unsupervised_avg10_img, gt_img, vmin, vmax)
        mae_unsupervised_avg20, mae_unsupervised_avg20_std = calc_mae_with_ref_window(unsupervised_avg20_img, gt_img, vmin, vmax)
        
        ssim_unsupervised, ssim_unsupervised_std = calc_ssim_with_ref_window(unsupervised_img, gt_img, vmin, vmax)
        ssim_unsupervised_avg10, ssim_unsupervised_avg10_std = calc_ssim_with_ref_window(unsupervised_avg10_img, gt_img, vmin, vmax)
        ssim_unsupervised_avg20, ssim_unsupervised_avg20_std = calc_ssim_with_ref_window(unsupervised_avg20_img, gt_img, vmin, vmax)
        
        lpips_unsupervised, _ = calc_lpips(unsupervised_img, gt_img, vmin, vmax)
        lpips_unsupervised_avg10, _ = calc_lpips(unsupervised_avg10_img, gt_img, vmin, vmax)
        lpips_unsupervised_avg20, _ = calc_lpips(unsupervised_avg20_img, gt_img, vmin, vmax)

        print('NFE: ', NFE)
        print('mae_avg10, ssim_avg10, lpips_avg10: ', mae_unsupervised_avg10, ssim_unsupervised_avg10, lpips_unsupervised_avg10)
        print('mae_avg20, ssim_avg20, lpips_avg20: ', mae_unsupervised_avg20, ssim_unsupervised_avg20, lpips_unsupervised_avg20)

        print('means: gt_mean, condition_mean, unsupervised_mean,  ', gt_mean, condition_mean,  unsupervised_mean)

        per_patient_results = per_patient_results + [mae_unsupervised, ssim_unsupervised, lpips_unsupervised, mae_unsupervised_avg10, ssim_unsupervised_avg10, lpips_unsupervised_avg10, mae_unsupervised_avg20, ssim_unsupervised_avg20, lpips_unsupervised_avg20]

    results.append(per_patient_results)
    column_list = ['patient_id', 'random_n']
    method_list = ['condition']#, 'noise2noise', 'noise2score', 'ddm2_final', 'supervised']
    for NFE in NFE_list:
        method_list = method_list + ['unsupervised_NFE'+str(NFE), 'unsupervised_avg10_NFE'+str(NFE), 'unsupervised_avg20_NFE'+str(NFE)]
    metric_list = ['mae', 'ssim', 'lpips']
    for method in method_list:
        for metric in metric_list:
            column_list = column_list + [metric + '_' + method]

    dd = pd.DataFrame(results, columns = column_list)  
    dd.to_excel(os.path.join('/host/d/projects/denoising/results', 'mayo_results_NFE_epoch285_bias200_eta1.0_epoch3050.xlsx'), index = False)

L310 0
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
mae_condition, ssim_condition, lpips_condition:  23.615942818438203 0.5635826699882492 0.16414213567972183
NFE is:  30
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
NFE:  30
mae_avg10, ssim_avg10, lpips_avg10:  14.10041062651847 0.6984004725226725 0.0560939583927393
mae_avg20, ssim_avg20, lpips_avg20:  13.664994184389393 0.7108111904715826 0.054675673320889476
means: gt_mean, condition_mean, unsupervised_mean,   -9.47522533458708 -8.77580332362904 -9.383683852803427
NFE is:

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
mae_condition, ssim_condition, lpips_condition:  18.00749451303126 0.696207412564322 0.0850381501019001
NFE is:  30
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
NFE:  30
mae_avg10, ssim_avg10, lpips_avg10:  12.100927240439312 0.7810378202024499 0.033224754147231576
mae_avg20, ssim_avg20, lpips_avg20:  11.771850430865728 0.7903041038682342 0.03423831805586815
means: gt_mean, condition_mean, unsupervised_mean,   12.010574505431084 11.79783304070518 11.833228694705987
NFE is:

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
mae_condition, ssim_condition, lpips_condition:  29.034950416706444 0.5563279310091798 0.1828617772459984
NFE is:  30
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
NFE:  30
mae_avg10, ssim_avg10, lpips_avg10:  17.203847666945144 0.6825182803762434 0.061652377024292944
mae_avg20, ssim_avg20, lpips_avg20:  16.653371950313044 0.6950759670859309 0.058701290786266326
means: gt_mean, condition_mean, unsupervised_mean,   -2.4335932367513755 -1.622108189920236 -2.101666007516902
NF

In [11]:
delta = unsupervised_avg20_mean - gt_mean
print('delta: ', delta)

delta:  5.196236961528198


In [14]:
unsupervised_avg_20_corrected = unsupervised_avg20_img - delta
unsupervised_avg_20_corrected_mean = np.mean(unsupervised_avg_20_corrected[(gt_img>-160) & (gt_img<240)])
print(unsupervised_avg_20_corrected_mean, unsupervised_avg_20_corrected.shape)

mae_unsupervised_avg20_corrected, mae_unsupervised_avg20_corrected_std = calc_mae_with_ref_window(unsupervised_avg_20_corrected, gt_img, vmin, vmax)
ssim_unsupervised_avg20_corrected, ssim_unsupervised_avg20_corrected_std = calc_ssim_with_ref_window(unsupervised_avg_20_corrected, gt_img, vmin, vmax)
lpips_unsupervised_avg20_corrected, _ = calc_lpips(unsupervised_avg_20_corrected, gt_img, vmin, vmax)
print('mae_avg20_corrected, ssim_avg20_corrected, lpips_avg20_corrected: ', mae_unsupervised_avg20_corrected, ssim_unsupervised_avg20_corrected, lpips_unsupervised_avg20_corrected)

-2.433593236751372 (512, 512, 50)
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
mae_avg20_corrected, ssim_avg20_corrected, lpips_avg20_corrected:  17.40215757619083 0.6808902301031337 0.06807499684393406
